In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from popsim import param_utils
from popsim.modules.tearing import Tearing, generate_disruption_phase_trajectory, generate_tearing_phase_trajectory

dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=4.0, dt=dt)
modes = [(2, 1), (3, 2)]
tearing_config = Tearing.Config(
    magx_time=time_base,
    modes=modes,
)

tearing_initial_state = Tearing.State(
    W={mode: 0.0 for mode in modes},
    F={mode: 0.0 for mode in modes},
    mode_phase={mode: 0.0 for mode in modes}
)

rot_dur = 1.0
locking_dur = 0.2
trigger_time = 2.0
disrupt_time = 3.5
dur_tq_to_spike = 1e-3

tearing_params = Tearing.Params(
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike, time_base, dt
    ),
    tearing_phase=generate_tearing_phase_trajectory(
        trigger_time, rot_dur, locking_dur, time_base, dt
    ),
)

tearing_module = Tearing(config=tearing_config)


In [ ]:
import jax

from popsim.modules.magnetic_diagnostics import LowNArray, load_lown_config

jax.config.update("jax_platforms", "cpu")

probe_connections, func_Bp_per_A = load_lown_config()

lown_array_config = LowNArray.Config(
    func_Bp_per_A=func_Bp_per_A,
    probe_connections=probe_connections,
    reconstructed_modes=[1,2,3]
)

lown_array_module = LowNArray(config=lown_array_config)

In [ ]:
from popsim.simulate import simulate
from popsim.simulators.tearing_sim.model import TearingSim

sim_config = TearingSim.Config(
    tearing_module=tearing_module,
    lown_array_module=lown_array_module,
)

sim_initial_state = TearingSim.State(
    tearing_state=tearing_initial_state
)

sim_params = TearingSim.Params(
    tearing_params=tearing_params
)

tearing_sim = TearingSim(config=sim_config)

sim_xarray = simulate(tearing_sim, time_base, sim_initial_state, sim_params)

In [ ]:
from popsim.visualize import visualize_time_series

#sim_xarray.to_netcdf("tearing_simulation.nc")
visualize_time_series(sim_xarray, max_cols=2)

In [ ]:
import matplotlib.pyplot as plt

mode_widths = [sim_xarray["state.tearing_state.W.(2, 1)"], sim_xarray["state.tearing_state.W.(3, 2)"]]
rotation_frequencies = [sim_xarray["state.tearing_state.F.(2, 1)"], sim_xarray["state.tearing_state.F.(3, 2)"]]
phases = [sim_xarray["state.tearing_state.mode_phase.(2, 1)"], sim_xarray["state.tearing_state.mode_phase.(3, 2)"]]
reconstructed_magnitudes = [sim_xarray["output.locals.reconstructed_magnitudes.1"], sim_xarray["output.locals.reconstructed_magnitudes.2"], sim_xarray["output.locals.reconstructed_magnitudes.3"]]
time = sim_xarray["time"].values

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for mode in range(len(mode_widths)):
    mode_width = mode_widths[mode]
    rotation_frequency = rotation_frequencies[mode]
    phase = phases[mode]
    reconstructed_magnitude = reconstructed_magnitudes[mode]
    axs[0,0].plot(time, rotation_frequency, label=f"{modes[mode]}")
    axs[0,1].plot(time, phase, label=f"{modes[mode]}")
    axs[1,0].plot(time, mode_width, label=f"{modes[mode]}")
    axs[1,1].plot(time, reconstructed_magnitude, label=f"n={mode+1}")

axs[1,1].plot(time, reconstructed_magnitudes[2], label="n=3")

axs[0,0].set_title("Rotation Frequency")
axs[0,0].set_xlabel("Time [s]")
axs[0,0].set_ylabel("Frequency [Hz]")
axs[0,0].legend()

axs[0,1].set_title("Mode Phase")
axs[0,1].set_xlabel("Time [s]")
axs[0,1].set_ylabel("Phase [rad]")
axs[0,1].legend()

axs[1,0].set_title("Mode Width")
axs[1,0].set_xlabel("Time [s]")
axs[1,0].set_ylabel("Width [m]")
axs[1,0].legend()

axs[1,1].set_title("Reconstructed Magnitudes")
axs[1,1].set_xlabel("Time [s]")
axs[1,1].set_ylabel("Magnitude [G]")
axs[1,1].legend()


plt.tight_layout()
plt.show()